In [1]:
import sys
sys.path.insert(0, '/Users/vahid/Downloads/FERNN-master/moving_mnist_fp')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings


print("✓ All imports successful")

✓ All imports successful


In [3]:
# Cell 1: Imports and Module Loading
import sys
import importlib.util
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Disable wandb if not initialized
os.environ['WANDB_ENABLED'] = '0'

# Load modules directly to avoid import issues
spec_dataset = importlib.util.spec_from_file_location(
    "moving_mnist_dataset",
    "/Users/vahid/Downloads/FERNN-master/moving_mnist_fp/moving_mnist_dataset.py"
)
moving_mnist_dataset = importlib.util.module_from_spec(spec_dataset)
spec_dataset.loader.exec_module(moving_mnist_dataset)
MovingMNISTDataset = moving_mnist_dataset.MovingMNISTDataset

spec_models = importlib.util.spec_from_file_location(
    "moving_mnist_models",
    "/Users/vahid/Downloads/FERNN-master/moving_mnist_fp/moving_mnist_models.py"
)
moving_mnist_models = importlib.util.module_from_spec(spec_models)
spec_models.loader.exec_module(moving_mnist_models)
SeqtoSeqRNN = moving_mnist_models.Seq2SeqFERNN

spec_utils = importlib.util.spec_from_file_location(
    "train_eval_utils",
    "/Users/vahid/Downloads/FERNN-master/moving_mnist_fp/train_eval_utils.py"
)
train_eval_utils = importlib.util.module_from_spec(spec_utils)
spec_utils.loader.exec_module(train_eval_utils)
train_epoch = train_eval_utils.train_epoch
eval_epoch = train_eval_utils.eval_epoch
eval_len_generalization = train_eval_utils.eval_len_generalization

spec_viz = importlib.util.spec_from_file_location(
    "visualization",
    "/Users/vahid/Downloads/FERNN-master/moving_mnist_fp/visualization.py"
)
visualization = importlib.util.module_from_spec(spec_viz)
spec_viz.loader.exec_module(visualization)
log_sequence_predictions_new = visualization.log_sequence_predictions_new

print("✓ All modules loaded successfully")

✓ All modules loaded successfully


In [5]:
# Cell 2: Configuration and Hyperparameters
# Modify these values to change training options

# Data options
root = './data'
seq_len = 20
input_frames = 10
batch_size = 8
image_size = 28
v_range = 2
data_v_range = 2
num_digits = 1

# Training options
epochs = 50
min_epochs = 50
lr = 1e-3
model_type = 'fernn'  # 'fernn' or 'grnn'
hidden_size = 128
num_layers = 1
kernel_size = 3
decoder_conv_layers = 4
max_train_samples = None
teacher_forcing_ratio = 0.0
grad_clip = 1.0

# Model options
pool_type = 'max'
use_differentiable_flow = False  # Set to True to use differentiable flow

# Evaluation options
gen_seq_len = 40
gen_vel_min = -2
gen_vel_max = 2
gen_vel_step = 1
gen_vel_n_seq = 128
run_velocity_generalization = False

# Wandb options (disabled by default)
wandb_entity = None
wandb_project = "FERNN"
wandb_dir = './tmp/'
wandb_name = None

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
print(f"Model: {model_type}")
print(f"Batch size: {batch_size}")
print(f"Hidden size: {hidden_size}")
print(f"V range: {v_range}")
print(f"Epochs: {epochs}")
print(f"LR: {lr}")
print(f"Teacher forcing: {teacher_forcing_ratio}")

Device: cpu
Model: fernn
Batch size: 8
Hidden size: 128
V range: 2
Epochs: 50
LR: 0.001
Teacher forcing: 0.0


In [6]:
# Cell 3: Data Loading
# Load and split datasets

train_dataset = MovingMNISTDataset(
    root=root,
    train=True,
    seq_len=seq_len,
    image_size=image_size,
    velocity_range_x=(-data_v_range, data_v_range),
    velocity_range_y=(-data_v_range, data_v_range),
    num_digits=num_digits
)

test_dataset = MovingMNISTDataset(
    root=root,
    train=False,
    seq_len=seq_len,
    image_size=image_size,
    velocity_range_x=(-data_v_range, data_v_range),
    velocity_range_y=(-data_v_range, data_v_range),
    num_digits=num_digits
)

gen_test_dataset = MovingMNISTDataset(
    root=root,
    train=False,
    seq_len=gen_seq_len,
    image_size=image_size,
    velocity_range_x=(-data_v_range, data_v_range),
    velocity_range_y=(-data_v_range, data_v_range),
    num_digits=num_digits,
    random=False
)

# Split training data into train and validation
val_size = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_ds, val_ds = random_split(train_dataset, [train_size, val_size])

# Limit training samples if specified
if max_train_samples is not None and max_train_samples < len(train_ds):
    train_ds = torch.utils.data.Subset(train_ds, torch.randperm(len(train_ds))[:max_train_samples])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
gen_test_loader = DataLoader(gen_test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}, Test size: {len(test_dataset)}")
print(f"Length-gen test size: {len(gen_test_dataset)} | Eval sequence length: {gen_seq_len}")
print(f"Input frames: {input_frames}, Pred frames: {seq_len - input_frames}")

Train size: 54000, Val size: 6000, Test size: 10000
Length-gen test size: 10000 | Eval sequence length: 40
Input frames: 10, Pred frames: 10


In [12]:
# Cell 4: Model Initialization
# Initialize model and optimizer

if model_type == "fernn":
    model = SeqtoSeqRNN(
        input_channels=1,
        hidden_channels=hidden_size,
        height=image_size,
        width=image_size,
        h_kernel_size=kernel_size,
        u_kernel_size=kernel_size,
        v_range=v_range,
        decoder_conv_layers=decoder_conv_layers
        ).to(device)
elif model_type == "grnn":
    assert v_range == 0, "v_range must be 0 for grnn"
    model = SeqtoSeqRNN(
        input_channels=1,
        hidden_channels=hidden_size,
        height=image_size,
        width=image_size,
        h_kernel_size=kernel_size,
        u_kernel_size=kernel_size,
        v_range=0,
        decoder_conv_layers=decoder_conv_layers
    ).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = torch.nn.MSELoss()

print(f"Model: {model_type}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Use differentiable flow: {use_differentiable_flow}")

Model: fernn
Parameters: 939,626
Use differentiable flow: False


In [10]:
# Cell 5: Training Loop
# Train the model with validation and save best model

history = {'train_loss': [], 'val_loss': [], 'test_loss': []}
best_val_loss = float('inf')

for epoch in range(1, epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, input_frames, teacher_forcing_ratio, grad_clip)
    val_loss = eval_epoch(model, val_loader, criterion, device, input_frames, epoch, split_name="val")
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    print(f"Epoch {epoch}/{epochs} | {model_type:^8} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    # Save model if it's the best so far
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model_filename = f"{model_type}_best_model_notebook.pth"
        model_path = os.path.join('./', model_filename)
        torch.save(model.state_dict(), model_path)
        print(f"Saved new best model at {model_path}")

# Load best model for testing
model.load_state_dict(torch.load(model_path))
model.eval()

print(f"\nBest model loaded from {model_path}")
print(f"Best validation loss: {best_val_loss:.4f}")

KeyboardInterrupt: 

In [11]:
# Cell 6: Testing
# Evaluate on test set and length generalization

test_loss = eval_epoch(model, test_loader, criterion, device, input_frames, epoch=0, split_name="test")
print(f"Test Loss: {test_loss:.4f}")

# Length generalization test
gen_mean, gen_std = eval_len_generalization(model, gen_test_loader, device, input_frames)
print(f"Length generalization mean MSE: {gen_mean.mean():.4f}")
print(f"Mean MSE per timestep (first 10): {gen_mean[:10]}")

# Optional: Velocity generalization if enabled
if run_velocity_generalization:
    # This would require implementing eval_velocity_generalization similar to train.py
    print("Velocity generalization not implemented in notebook yet")
    pass

KeyboardInterrupt: 

In [ ]:
# Cell 7: Visualization
# Visualize predictions on test data

# Get a batch from test loader
test_batch = next(iter(test_loader))
seq = test_batch[0].to(device)  # Extract sequence tensor
input_seq = seq[:2, :input_frames]  # First 2 samples
target_seq = seq[:2, input_frames:]

# Generate predictions
with torch.no_grad():
    pred_seq = model(input_seq, pred_len=target_seq.size(1), teacher_forcing_ratio=0.0)
    if isinstance(pred_seq, tuple):
        pred_seq = pred_seq[0]  # Extract predictions if tuple

# Visualize using matplotlib (since wandb is disabled)
fig, axes = plt.subplots(3, target_seq.size(1), figsize=(15, 6))
for t in range(target_seq.size(1)):
    # Ground truth
    axes[0, t].imshow(target_seq[0, t, 0].cpu().numpy(), cmap='gray')
    axes[0, t].set_title(f'GT t={t+1}')
    axes[0, t].axis('off')
    
    # Prediction
    axes[1, t].imshow(pred_seq[0, t, 0].cpu().numpy(), cmap='gray')
    axes[1, t].set_title(f'Pred t={t+1}')
    axes[1, t].axis('off')
    
    # Difference
    diff = torch.abs(pred_seq[0, t] - target_seq[0, t]).cpu().numpy()
    axes[2, t].imshow(diff[0], cmap='hot', vmin=0, vmax=1)
    axes[2, t].set_title(f'Diff t={t+1}')
    axes[2, t].axis('off')

plt.tight_layout()
plt.show()

print("Visualization complete. Modify the batch index or number of samples as needed.")